# Notebook 3 — Train / Validation / Test Split

**Goal:** split the data BEFORE any deep analysis, so the test set never leaks
into your decisions.

In [1]:
import pandas as pd
import numpy as np

ARTIFACTS_DIR = "artifacts"
labeled = pd.read_parquet(f"{ARTIFACTS_DIR}/02_labeled_table.parquet")
labeled['order_purchase_timestamp'] = pd.to_datetime(labeled['order_purchase_timestamp'])

print("Date range:", labeled['order_purchase_timestamp'].min(), "->",
      labeled['order_purchase_timestamp'].max())
print(labeled['is_late'].value_counts(normalize=True))

Date range: 2016-09-15 12:16:38 -> 2018-08-29 15:00:37
is_late
0    0.918871
1    0.081129
Name: proportion, dtype: float64


## Random vs time-based split — think about it

- **Random split**: fine if you assume the model will be retrained often and
  the delivery process is roughly stable over time. Easiest to keep the label
  ratio balanced across splits (stratify).
- **Time-based split** (train = earliest orders, test = most recent orders):
  more realistic for a production pipeline that predicts on *future* orders —
  avoids the model "seeing the future" (seasonality, courier changes, etc.)
  that a random split can hide.

Given this is a delivery-time prediction problem meant to run in production on
new incoming orders, a **time-based split is the more honest choice** — but a
stratified random split is documented below too so you can compare. Pick the
one you can justify and say why in your write-up.

In [2]:
# ---- Option A: time-based split (recommended for this problem) ----
labeled_sorted = labeled.sort_values('order_purchase_timestamp').reset_index(drop=True)

n = len(labeled_sorted)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_time = labeled_sorted.iloc[:train_end]
val_time   = labeled_sorted.iloc[train_end:val_end]
test_time  = labeled_sorted.iloc[val_end:]

for name, split in [("train", train_time), ("val", val_time), ("test", test_time)]:
    print(f"{name:5s} n={len(split):6d}  "
          f"dates {split['order_purchase_timestamp'].min().date()} -> "
          f"{split['order_purchase_timestamp'].max().date()}  "
          f"late%={split['is_late'].mean()*100:.2f}")

train n= 67533  dates 2016-09-15 -> 2018-04-15  late%=9.03
val   n= 14471  dates 2018-04-15 -> 2018-06-21  late%=5.34
test  n= 14472  dates 2018-06-21 -> 2018-08-29  late%=6.61


In [3]:
# ---- Option B: stratified random split (keeps label ratio identical) ----
from sklearn.model_selection import train_test_split

train_rand, temp_rand = train_test_split(
    labeled, test_size=0.30, stratify=labeled['is_late'], random_state=42
)
val_rand, test_rand = train_test_split(
    temp_rand, test_size=0.50, stratify=temp_rand['is_late'], random_state=42
)

for name, split in [("train", train_rand), ("val", val_rand), ("test", test_rand)]:
    print(f"{name:5s} n={len(split):6d}  late%={split['is_late'].mean()*100:.2f}")

train n= 67533  late%=8.11
val   n= 14471  late%=8.11
test  n= 14472  late%=8.11


## Choose ONE split strategy to move forward with
Set `USE_TIME_SPLIT = True/False` below and stick with it for every notebook
after this one.

In [4]:
USE_TIME_SPLIT = True   # <-- your decision, justify it in your write-up

if USE_TIME_SPLIT:
    train, val, test = train_time, val_time, test_time
else:
    train, val, test = train_rand, val_rand, test_rand

print("Final split sizes:", len(train), len(val), len(test))
print("Late % per split:", train['is_late'].mean(), val['is_late'].mean(), test['is_late'].mean())

Final split sizes: 67533 14471 14472
Late % per split: 0.0902817881628241 0.053417179185958126 0.06612769485903815


## Artifact: train, validation, and test files

In [5]:
train.to_parquet(f"{ARTIFACTS_DIR}/03_train.parquet", index=False)
val.to_parquet(f"{ARTIFACTS_DIR}/03_val.parquet", index=False)
test.to_parquet(f"{ARTIFACTS_DIR}/03_test.parquet", index=False)
print("Saved train/val/test to", ARTIFACTS_DIR)

Saved train/val/test to artifacts
